# Dataspace Contract Negotiation (DSP 2025-1)

This notebook executes the complete Dataspace Protocol (DSP 2025-1) Contract Negotiation lifecycle between Consumer and Provider:
1. Executes prerequisites (`setup_catalog_policies.ipynb`) if catalog and policy targets are not already loaded.
2. Initiates negotiation request from Consumer to Provider (`/dsp/current/negotiations/rpc/setup-request-init`).
3. Dispatches initial offer from Provider to Consumer (`/dsp/current/negotiations/rpc/setup-offer`).
4. Submits counter-request based on offer from Consumer to Provider (`/dsp/current/negotiations/rpc/setup-request`).
5. Updates and confirms offer from Provider to Consumer (`/dsp/current/negotiations/rpc/setup-offer`).
6. Accepts offer from Consumer to Provider (`/dsp/current/negotiations/rpc/setup-acceptance`).
7. Dispatches binding contract agreement from Provider to Consumer (`/dsp/current/negotiations/rpc/setup-agreement`).
8. Verifies contract agreement from Consumer to Provider (`/dsp/current/negotiations/rpc/setup-verification`).
9. Finalizes negotiation from Provider to Consumer (`/dsp/current/negotiations/rpc/setup-finalization`).
10. Extracts and displays finalized contract agreement details (`agreement_id`).

## 1. Prerequisites
Ensure that catalog and policies are populated and target identifiers are available. If not already loaded in the environment, `setup_catalog_policies.ipynb` is automatically executed.

In [ ]:
if "target_id" not in globals() or not target_id:
    %run ./setup_catalog_policies.ipynb

## 2. Negotiation Request Initiation (Consumer -> Provider)
The Consumer initiates contract negotiation by sending a `ContractRequestMessage` via `/dsp/current/negotiations/rpc/setup-request-init` referencing the target dataset and offer policy.
The Provider assigns a `providerPid` and transitions the negotiation state to `REQUESTED`.

In [2]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:d187ead6-9488-4dcd-b0f6-bd89e6a138ad",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
  

## 3. Provider Initial Offer (Provider -> Consumer)
The Provider responds with a counter-offer by sending a `ContractOfferMessage` via `/dsp/current/negotiations/rpc/setup-offer`.
The negotiation transitions to `OFFERED`.

In [3]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:d187ead6-9488-4dcd-b0f6-bd89e6a138ad",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:aecfd1a5-7d2c-4bc0-b447-b6045d77f43c"
    },
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:76330add-dd74-4fe5-85

## 4. Consumer Counter-Request Based on Offer (Consumer -> Provider)
The Consumer reviews the offer and submits a counter-request via `/dsp/current/negotiations/rpc/setup-request`.
The negotiation state transitions back to `REQUESTED`.

In [4]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:d187ead6-9488-4dcd-b0f6-bd89e6a138ad",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:aecfd1a5-7d2c-4bc0-b447-b6045d77f43c"
    },
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bb6a689c-bbf4-4087-b725-91c83b9f71e5",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgen

## 5. Provider Offer Confirmation (Provider -> Consumer)
The Provider confirms or updates the terms of the offer via `/dsp/current/negotiations/rpc/setup-offer`.
The negotiation state transitions to `OFFERED`.

In [5]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:d187ead6-9488-4dcd-b0f6-bd89e6a138ad",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:aecfd1a5-7d2c-4bc0-b447-b6045d77f43c"
    },
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:76330add-dd74-4fe5-85ca-19ca9b16265e",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "OFFERED",
    "stateAttribute": null,
    "associated

## 6. Consumer Offer Acceptance (Consumer -> Provider)
The Consumer accepts the Provider's offer terms via `/dsp/current/negotiations/rpc/setup-acceptance`.
The negotiation state transitions to `ACCEPTED`.

In [6]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bb6a689c-bbf4-4087-b725-91c83b9f71e5",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdH

## 7. Provider Contract Agreement (Provider -> Consumer)
The Provider generates the binding `ContractAgreementMessage` containing the finalized agreement terms and sends it to the Consumer via `/dsp/current/negotiations/rpc/setup-agreement`.
The negotiation state transitions to `AGREED`.

In [7]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:76330add-dd74-4fe5-85ca-19ca9b16265e",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQ

## 8. Consumer Agreement Verification (Consumer -> Provider)
The Consumer verifies receipt and terms of the contract agreement by sending a `ContractAgreementVerificationMessage` via `/dsp/current/negotiations/rpc/setup-verification`.
The negotiation state transitions to `VERIFIED`.

In [8]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:bb6a689c-bbf4-4087-b725-91c83b9f71e5",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdH

## 9. Provider Negotiation Finalization (Provider -> Consumer)
The Provider acknowledges verification and sends a `ContractNegotiationEventMessage` (finalization) via `/dsp/current/negotiations/rpc/setup-finalization`.
The negotiation transitions to the terminal state `FINALIZED`.

In [9]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:26a9413f-bea3-446a-b023-e493e73341f3",
    "providerPid": "urn:provider-pid:fb5c8e0a-095d-4be2-ac3c-b04e444b099d",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:76330add-dd74-4fe5-85ca-19ca9b16265e",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd

## 10. Final Agreement Details
Display the finalized contract agreement and extract the `agreement_id` required for subsequent transfer requests.

In [10]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:ed676de1-2364-47c1-b8af-cb019ace23b4",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public",
  "negotiationAgentProcessId": "urn:negotiation-process:76330add-dd74-4fe5-85ca-19ca9b16265e",
  "negotiationAgentMessageId": "urn:negotiation-message:8106a1a4-cea2-44f3-aa61-d194e0f0de55",
  "consumerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "providerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXN